# Random Forest（2001）

- **核心原理**：Bagging＋隨機特徵
- **變體**：Bagging（不挑特徵，只做 bootstrap）、Extra Trees（連切點都隨機挑）
- **推薦資料集**：Adult（Census Income；1994 年美國人口普查的 48,842 人；目標：年收入是否 > 50K 美元）

本章依照 README 的「每章標準流程」：EDA → Baseline → 手寫模型 → 套件實作 → 調參 → 測試集評估 → 診斷 → 本章小結。

第五章的結論是：決策樹好解釋、不用標準化、會自動處理交互作用，但**變異很高** —— 換一批訓練資料，第二層以下就整個變了（實驗 4：兩棵 bootstrap 樹的預測有 21% 不同），而把 20 棵樹平均起來，test accuracy 就從 0.746 升到 0.790。

Random Forest 就是把這個觀察做到底：

1. **Bagging**（bootstrap aggregating）：在很多個 bootstrap 樣本上各長一棵**完全長滿**的樹（低偏差、高變異），再平均 → 降低變異；
2. **隨機特徵**：每個節點只從隨機挑的幾個特徵裡找最好的切法 → 讓樹與樹「更不像」，平均的效果更好。

這一章的重點：**平均為什麼能降低變異、樹之間的「相關」為什麼是關鍵、OOB（out-of-bag）為什麼是免費的驗證集、特徵重要度要怎麼看才可信。**

### 閱讀指南：本章的符號與跨章引用

下一格是「前置複習」；第 2、3A、3B、3C、3D、4、5、7 節開頭另有各自的符號表或名詞表。

**資料**：每一列是一個人。數值欄：`age`、`fnlwgt`（人口普查的抽樣權重：這筆資料代表多少人）、`education-num`（教育年數）、`capital-gain`／`capital-loss`（資本利得／損失）、`hours-per-week`。類別欄：`workclass`、`education`、`marital-status`、`occupation`、`relationship`、`race`、`sex`、`native-country`。目標 `class`：`>50K` = 1、`<=50K` = 0。

**本章最常出現的符號**（各小節開頭有完整符號表）：

| 符號 | 意義 | 本章的值 |
| --- | --- | --- |
| $N$ | 訓練樣本數 | 約 39,000（以載入結果為準）；手寫版用 5,000 筆的子樣本 |
| $d$ | 特徵數（類別欄用 ordinal 編碼後） | 14 |
| $B$ | 樹的棵數（`n_estimators`） | — |
| $T_b$ | 第 $b$ 棵樹；$\hat p_b(x)$ 是它對 $x$ 的預測機率 | — |
| $\hat p(x)$ | 森林的預測 $= \frac{1}{B}\sum_b \hat p_b(x)$ | — |
| $m$ | 每個節點隨機挑的特徵數（`max_features`）；$m = d$ 就是 bagging | 預設 $\sqrt{d} \approx 4$ |
| bootstrap 樣本 | 從 $N$ 筆中抽後放回抽 $N$ 次 | — |
| OOB | out-of-bag：某棵樹的 bootstrap 樣本**沒抽到**的那些資料 | 每棵樹約 36.8% |
| $\rho$ | 兩棵樹的預測之間的相關係數 | — |
| $\sigma^2$ | 一棵樹的預測的變異數（換訓練資料時會變多少） | — |

**跨章引用對照**：

| 文中寫法 | 指的是 | 那裡的結論（重述） |
| --- | --- | --- |
| 第五章 | 第一部分第 5 節：決策樹（`01-classic-ml/05-decision-tree`） | CART：每個節點在所有特徵、所有切點中找 Gini 下降最多的一刀（排序 + `cumsum` 一次算完所有切點）；遞迴長樹；預測 = 葉子的類別比例。 |
| 第五章實驗 4 | 同上，TODO 15 | 20 棵 bootstrap 樹的根節點都一樣（sex），第二層開始不同；兩兩預測不一致 21%；平均 20 棵的機率，accuracy 0.746 → 0.790。 |
| 第五章的特徵重要度 | 同上，TODO 10、20 | impurity importance 用**訓練資料**算，會偏向「值很多」的特徵：純雜訊的連續欄在長滿的樹上排第二；permutation importance（用測試資料打亂一欄）不會被騙。 |
| 第五章「單調轉換」 | 同上，TODO 14 | 樹只看排序，不需要標準化。 |
| 第三章的不平衡資料 | 邏輯回歸 | 類別不平衡時 accuracy 會誤導；看 ROC AUC、少數類的 precision／recall。 |
| 第六章的校準 | 樸素貝氏，TODO 12 | 把預測依信心分組，比較「平均信心」與「實際猜對的比例」。 |
| 第八章的重複 CV | SVM | 資料少時用 `RepeatedStratifiedKFold`；本章資料多，單次 CV 就夠穩。 |

### 前置複習：本章會用到的舊知識

#### 數學

**1. 平均的變異數**：$B$ 個**獨立**、變異數都是 $\sigma^2$ 的隨機變數，平均的變異數是 $\sigma^2 / B$。例：擲一顆骰子的變異數約 2.92；擲 4 顆取平均，變異數約 0.73。

**2. 有相關時的平均**：如果兩兩之間的相關係數都是 $\rho$，
$$\text{Var}\left(\frac{1}{B}\sum_{b=1}^{B} Z_b\right) = \rho\,\sigma^2 + \frac{1 - \rho}{B}\,\sigma^2$$
$B \to \infty$ 時第二項消失，但**第一項 $\rho\sigma^2$ 永遠降不下去**。例：$\sigma^2 = 1$、$\rho = 0.5$ → 不管平均多少棵，變異數至少 0.5；$\rho = 0.1$ → 至少 0.1。**想讓平均更有效，就要降低 $\rho$**（這就是隨機特徵的目的）。

**3. 相關係數**：$\rho(U, V) = \dfrac{\text{Cov}(U, V)}{\sigma_U \sigma_V} \in [-1, 1]$；`np.corrcoef(u, v)[0, 1]`。

**4. Bootstrap**（第五章前置複習 7 的重述）：抽後放回抽 $N$ 次，某一筆**沒被抽中**的機率 $(1 - 1/N)^N \to e^{-1} \approx 0.368$。所以每個 bootstrap 樣本約含 63.2% 的不同資料（其餘是重複），剩下約 36.8% 就是這棵樹的 **OOB** 資料 —— 這棵樹完全沒看過它們。

**5. 多數決的威力（Condorcet）**：$B$ 個**獨立**的投票者，每人答對機率 $p > 0.5$，多數決答對的機率隨 $B$ 增加趨近 1。例：$p = 0.6$、$B = 25$ → 約 0.85。**但前提是「獨立」**；投票者想法都一樣時，多數決 = 一個人。

**6. 偏差—變異**（重述）：平均很多個模型**不會降低偏差**（每個都偏一樣多，平均還是偏），只降低變異 → 適合拿「低偏差、高變異」的模型（長滿的樹）來平均。

#### 程式

**7. `rng = np.random.default_rng(seed)`**：
```python
rng.integers(0, N, size=N)              # 抽後放回：可能重複
rng.choice(d, size=m, replace=False)    # 不放回：m 個不同的特徵編號
np.bincount(idx, minlength=N)           # 每一筆被抽到幾次；== 0 的就是 OOB
```

**8. 物件的串列**：`self.trees_ = []`，迴圈裡 `self.trees_.append(tree)`；之後 `for tree in self.trees_:` 拿到的是**每一棵樹物件**（不是編號）。`zip(self.trees_, self.oob_masks_)` 一次走兩個平行的串列。

**9. 沿著新的一軸疊起來再平均**：`np.stack([a, b, c])` 把三個 $(n, K)$ 疊成 $(3, n, K)$；`.mean(axis=0)` → $(n, K)$。

**10. 累加器 + 計數器**：要算「只在某些時候才有值」的平均，用兩個陣列：`total += value * mask`、`count += mask`，最後 `total / count`（`count = 0` 的位置要另外處理）。

**11. `OrdinalEncoder`**：把類別換成 0, 1, 2, …（依字母順序）。對樹來說，這等於把類別排成一個（任意的）順序，再用「≤ 門檻」切；第五章用 one-hot，這裡為了讓手寫版快一點、也讓 $d$ 小一點，用 ordinal。

---
## 1. 資料與探索性資料分析（EDA）

觀察直接寫在每格程式最上面的註解（`# 觀察：`）。

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from ml_dl_llm.data import load_adult

X_train, X_test, y_train, y_test = load_adult()


def as_label(y) -> pd.Series:
    """載入結果 → 0／1：'>50K'（或 '>50K.'）= 1。數值標籤則直接轉整數。"""
    y = pd.Series(y)
    if pd.api.types.is_numeric_dtype(y):
        return y.astype(int)
    return y.astype(str).str.contains(">").astype(int)


y_train, y_test = as_label(y_train), as_label(y_test)
y_train.index, y_test.index = X_train.index, X_test.index
NUM = [c for c in X_train.columns if pd.api.types.is_numeric_dtype(X_train[c])]
CAT = [c for c in X_train.columns if c not in NUM]
train = X_train.assign(income=y_train)

print("train / test：", X_train.shape, X_test.shape)
print("數值欄：", NUM)
print("類別欄：", CAT)
display(X_train.head())

In [ ]:
# 類別分布、缺值、類別欄的不同值個數
# 觀察：（>50K 佔多少？「全部猜 <=50K」的 accuracy 是多少？缺值在哪幾欄、多少？哪個類別欄的類別最多？）
display(pd.DataFrame({"train": y_train.value_counts(normalize=True), "test": y_test.value_counts(normalize=True)}).round(3))
info = pd.DataFrame({"缺值比例": X_train.isna().mean().round(4), "不同值個數": X_train.nunique()})
display(info.T)

In [ ]:
# 數值欄的分布
# 觀察：（capital-gain、capital-loss 有多少比例是 0？最大值多少？fnlwgt 的不同值有多少個、和收入有關嗎？
#        education-num 和 education 是不是同一件事？）
display(X_train[NUM].describe().T.round(1).assign(zero_ratio=(X_train[NUM] == 0).mean().round(3)))
fig, axes = plt.subplots(1, 4, figsize=(16, 3.2))
for ax, col in zip(axes, ["age", "hours-per-week", "education-num", "fnlwgt"]):
    for lab, name in [(0, "<=50K"), (1, ">50K")]:
        ax.hist(X_train.loc[y_train == lab, col], bins=40, alpha=0.6, density=True, label=name)
    ax.set_title(col)
axes[0].legend()
fig.tight_layout()
plt.show()
if "education" in X_train:
    display(pd.crosstab(X_train["education"], X_train["education-num"]).replace(0, "").iloc[:, :8])

In [ ]:
# 類別欄與收入
# 觀察：（哪幾個類別欄把 >50K 的比例拉得最開？relationship 和 marital-status 是不是在講同一件事？
#        native-country 的類別雖多，大部分人是哪一國？）
fig, axes = plt.subplots(2, 3, figsize=(16, 7))
for ax, col in zip(axes.ravel(), ["relationship", "marital-status", "education", "occupation", "workclass", "sex"]):
    rate = train.groupby(col, observed=True)["income"].agg(["mean", "size"]).sort_values("mean")
    ax.barh(rate.index.astype(str), rate["mean"], color="#4C72B0")
    ax.set_title(f">50K rate by {col}")
    ax.tick_params(axis="y", labelsize=8)
fig.tight_layout()
plt.show()
print("native-country 前 3：", X_train["native-country"].value_counts(normalize=True).head(3).round(3).to_dict())

In [ ]:
# TODO 1（問答）EDA 小結
#
# (1) 不平衡：>50K 佔多少？「全部猜 <=50K」的 accuracy 是多少？所以本章選模型時除了 accuracy，還要看哪個指標？
# 答：
#
# (2) 冗餘與可疑的特徵：
#     (a) education 和 education-num 是同一件事嗎？relationship 和 marital-status 呢？
#         這種「兩欄講同一件事」的特徵，對一棵樹、對一片森林（每個節點只能看隨機的幾欄）各有什麼影響？
#     (b) fnlwgt 是抽樣權重（這筆資料代表多少人），不是這個人的屬性。它的不同值有幾個？跟收入的分布看起來有關嗎？
#         先預測：它在 impurity importance 裡會排得很前面還是很後面？（第五章 TODO 20 (2)；第 4 節會驗證）
# 答：
#
# (3) capital-gain 大多是 0、少數很大。對樹來說這需要處理嗎？（第五章：樹只看排序）
# 答：

---
## 2. 前處理與 Baseline

#### 前處理

- **類別欄**：缺值補成 `"Missing"`（缺值本身當一個類別），再 `OrdinalEncoder`（前置複習 11）；測試集出現沒看過的類別編成 −1。
- **數值欄**：不動（樹不需要標準化）。
- 全部包進 `ColumnTransformer`，只用訓練集 fit。前處理後 $d = 14$。

#### Baseline

- **Dummy prior**。
- **單一決策樹**（第五章）：完全長滿，以及 `max_depth=8`（一棵「有剪枝」的樹）。

#### 本節名詞

- **ROC AUC**：用預測機率排序，隨機挑一個正例、一個負例，正例分數比較高的機率；跟門檻無關，0.5 = 亂猜。不平衡資料上比 accuracy 有資訊。
- **5-fold 分層 CV**：資料多（約 39,000 筆），單次 5-fold 就很穩，不需要重複（對照第八章）。

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.tree import DecisionTreeClassifier


def make_prep() -> ColumnTransformer:
    """原始 DataFrame → 14 個數值欄的 DataFrame（類別欄 ordinal 編碼，缺值 → "Missing"）。"""
    cat_pipe = Pipeline(
        [
            ("imp", SimpleImputer(strategy="constant", fill_value="Missing")),
            ("ord", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
        ]
    )
    return ColumnTransformer([("cat", cat_pipe, CAT)], remainder="passthrough", verbose_feature_names_out=False).set_output(transform="pandas")


prep = make_prep().fit(X_train)
Z_train, Z_test = prep.transform(X_train), prep.transform(X_test)
FEATURES = Z_train.columns.tolist()
Xtr, Xte = Z_train.to_numpy(dtype=float), Z_test.to_numpy(dtype=float)
ytr, yte = y_train.to_numpy(), y_test.to_numpy()
print("特徵：", FEATURES)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
SCORING = {"accuracy": "accuracy", "roc_auc": "roc_auc", "f1": "f1"}


def cv_summary(model, X=X_train, y=y_train) -> dict[str, str]:
    """5-fold CV：accuracy、ROC AUC、F1 的「平均 ± 標準差」，以及平均 fit 秒數。"""
    s = cross_validate(model, X, y, cv=cv, scoring=SCORING, n_jobs=-1)
    out = {k: f"{s['test_' + k].mean():.4f} ± {s['test_' + k].std():.4f}" for k in SCORING}
    out["fit s"] = f"{s['fit_time'].mean():.2f}"
    return out


baseline = pd.DataFrame(
    {
        "Dummy prior": cv_summary(DummyClassifier(strategy="prior")),
        "Tree fully grown": cv_summary(Pipeline([("prep", make_prep()), ("tree", DecisionTreeClassifier(random_state=0))])),
        "Tree max_depth=8": cv_summary(Pipeline([("prep", make_prep()), ("tree", DecisionTreeClassifier(max_depth=8, random_state=0))])),
    }
).T
display(baseline)

In [ ]:
# TODO 2（問答）
#
# (1) Dummy 的 ROC AUC 是多少？為什麼？
# 答：
#
# (2) 長滿的樹 vs 深度 8 的樹：
#     (a) accuracy、AUC 各差多少？哪一個 AUC 比較差得多？為什麼長滿的樹的 AUC 特別吃虧？
#         （提示：長滿的樹的葉子大多只有 1–3 筆 → predict_proba 大多是 0 或 1 → 用機率「排序」時會怎樣？第五章 TODO 16 (2)）
#     (b) 第五章用剪枝（限制深度、ccp_alpha）降低變異。Random Forest 走另一條路：「不剪、長滿，但長很多棵再平均」。
#         先預測：一片長滿的樹組成的森林，AUC 會比深度 8 的單棵樹高還是低？
# 答：

---
## 3. 手寫模型

| 小節 | 做什麼 | TODO |
| --- | --- | --- |
| 3A | 第五章的決策樹（完整提供）＋**每個節點隨機挑特徵** | 4 |
| 3B | Bootstrap 與 OOB | 3 |
| 3C | 森林：長很多棵、平均、OOB 分數 | 5、6、7 |
| 3D | 實驗：樹的棵數、隨機特徵的數量與樹之間的相關、變異、Extra Trees | — |

手寫的樹是 Python 迴圈，在 39,000 筆上長 50 棵太慢；**手寫版用 5,000 筆的隨機子樣本**，3D 的實驗與後面各節用 sklearn 跑完整資料。

### 3A. 決策樹（第五章）＋ 隨機特徵

下一格是第五章的 `DecisionTree`（`gini`、`_split_stats`、`best_split_on_feature`、`_grow`、`apply_one`、`predict_proba`、`feature_importances_`），已經完整寫好，只多了兩個參數：

- `max_features`：每個節點隨機挑幾個特徵來比較；`None` = 全部（第五章的行為）。
- `random_state`：這棵樹自己的亂數種子。

**唯一要改的地方是 `best_split`**：第五章走過全部 $d$ 個特徵；這裡**每個節點重新**隨機挑 $m$ 個不同的特徵，只在它們之中找最好的一刀（TODO 4）。注意是「每個節點」重挑，不是「每棵樹」挑一次。

#### 符號表（3A）

| 符號 | 意義 | 程式裡 |
| --- | --- | --- |
| $d$ | 特徵數 | `self.n_features_in_` |
| $m$ | 每個節點比較的特徵數 | `self._m`（由 `max_features` 換算：`"sqrt"` → $\lfloor\sqrt{d}\rfloor$、整數直接用、`None` → $d$） |
| candidates | 這個節點隨機挑到的特徵編號，$m$ 個、不重複 | `candidates` |
| `self._rng` | 這棵樹的亂數產生器（fit 時用 `random_state` 建立） | `self._rng` |

#### 最小例子：TODO 4（每個節點隨機挑 $m$ 個特徵）

$d = 6$、$m = 2$，假裝有 3 個節點，每個節點挑一次。

1. **正確答案**：用 Python 的 `random.sample` 挑。重點：每個節點挑的都是 2 個**不同**的特徵，而且三個節點挑的不一樣。
2. **積木**：`rng.integers`（會重複）vs `rng.choice(..., replace=False)`（不會重複）；`np.arange(d)`；`m = d` 時 `rng.choice` 只是把順序打亂。
3. **留給你**：在 `best_split` 裡，`range(d)` 換成什麼？

In [ ]:
# 最小例子：TODO 4（每個節點隨機挑 m 個特徵）
import random

d_mini, m_mini = 6, 2

# ---- 1. 正確答案：random.sample ----
random.seed(0)
for node in range(3):
    print(f"節點 {node}：random.sample(range(6), 2) =", sorted(random.sample(range(d_mini), m_mini)))

# ---- 2. 積木 ----
rng_mini = np.random.default_rng(0)
print("\nrng.integers(0, 6, size=2)          ", [rng_mini.integers(0, d_mini, size=m_mini).tolist() for _ in range(5)], "← 可能挑到同一個特徵兩次")
print("rng.choice(6, size=2, replace=False)", [sorted(rng_mini.choice(d_mini, size=m_mini, replace=False).tolist()) for _ in range(5)], "← 一定不重複")
print("m = d 時：rng.choice(6, 6, replace=False) =", rng_mini.choice(d_mini, d_mini, replace=False), "← 全部都有，只是順序亂了（所以 m = d 時乾脆不用亂數，見 TODO 8 (1)）")

# ---- 3. 留給你 ----
# 問題：best_split 的迴圈 for j in ...：要走過哪些特徵？亂數產生器用哪一個（self._rng）？挑幾個（self._m）？
# 寫完 TODO、執行過下一格之後，取消下面的註解來對照（3 個節點各挑 2 個）：
# t_mini = DecisionTree(max_features=2, random_state=0); t_mini.n_features_in_ = 6; t_mini._setup_random(); print([sorted(t_mini._candidate_features().tolist()) for _ in range(3)])

In [ ]:
from dataclasses import dataclass
from typing import Literal, Self

from numpy.typing import ArrayLike


def gini(counts: ArrayLike) -> np.ndarray:
    """Gini 不純度。counts (..., K) → (...,)（第五章 TODO 3）。"""
    counts = np.asarray(counts, dtype=float)
    n = counts.sum(axis=-1, keepdims=True)
    p = np.divide(counts, n, out=np.zeros_like(counts), where=n > 0)
    return (p * (1 - p)).sum(axis=-1)


@dataclass
class Node:
    """樹的一個節點（第五章）。葉子：left = right = None。"""

    n_samples: int
    value: np.ndarray
    impurity: float
    depth: int
    feature: int = -1
    threshold: float = np.nan
    left: "Node | None" = None
    right: "Node | None" = None

    @property
    def is_leaf(self) -> bool:
        return self.left is None


class DecisionTree:
    """第五章的 CART 分類樹（Gini），多了 max_features 與 random_state。"""

    def __init__(self, max_depth: int | None = None, min_samples_leaf: int = 1, max_features: int | Literal["sqrt"] | None = None, random_state: int | None = None) -> None:
        self.max_depth = max_depth
        self.min_samples_leaf = min_samples_leaf
        self.max_features = max_features
        self.random_state = random_state

    # ---- 第五章：一個特徵上所有切點的 ΔI（排序 + cumsum）----
    def best_split_on_feature(self, x: np.ndarray, y_idx: np.ndarray, K: int) -> tuple[float, float]:
        n = len(x)
        if n < 2:
            return -np.inf, np.nan
        order = np.argsort(x, kind="stable")
        xs, onehot = x[order], np.eye(K)[y_idx[order]]
        left = np.cumsum(onehot, axis=0)[:-1]
        total = onehot.sum(axis=0)
        n_left = np.arange(1, n)
        gain = gini(total) - (n_left * gini(left) + (n - n_left) * gini(total - left)) / n
        valid = (xs[:-1] < xs[1:]) & (n_left >= self.min_samples_leaf) & (n - n_left >= self.min_samples_leaf)
        if not valid.any():
            return -np.inf, np.nan
        i = int(np.argmax(np.where(valid, gain, -np.inf)))
        return float(gain[i]), float((xs[i] + xs[i + 1]) / 2)

    # ---- 本章新增：隨機特徵 ----
    def _setup_random(self) -> None:
        """依 max_features 算出 m，並建立這棵樹的亂數產生器。"""
        d = self.n_features_in_
        if self.max_features is None:
            self._m = d
        elif self.max_features == "sqrt":
            self._m = max(1, int(np.sqrt(d)))
        else:
            self._m = int(self.max_features)
        self._rng = np.random.default_rng(self.random_state)

    def _candidate_features(self) -> np.ndarray:
        """這個節點要比較的特徵編號 (m,)，不重複。"""
        if self._m >= self.n_features_in_:
            return np.arange(self.n_features_in_)  # 全部特徵、固定順序 = 第五章的行為（TODO 8 (1)）
        # TODO 4（程式）：每個節點隨機挑 self._m 個不同的特徵
        #   - 用 self._rng.choice(總數, size=個數, replace=False)
        raise NotImplementedError("TODO 4：隨機挑特徵")

    def best_split(self, X: np.ndarray, y_idx: np.ndarray) -> tuple[float, int, float]:
        best = (-np.inf, -1, np.nan)
        for j in self._candidate_features():  # 第五章是 range(d)
            g, s = self.best_split_on_feature(X[:, j], y_idx, self.n_classes_)
            if g > best[0] + 1e-12:
                best = (g, int(j), s)
        return best

    # ---- 第五章：遞迴長樹、預測、重要度 ----
    def fit(self, X: ArrayLike, y: ArrayLike) -> Self:
        self.X_ = np.asarray(X, dtype=float)
        self.classes_, self.y_idx_ = np.unique(np.asarray(y), return_inverse=True)
        self.n_classes_, self.n_features_in_ = len(self.classes_), self.X_.shape[1]
        self._setup_random()
        self.root_ = self._grow(np.arange(len(self.X_)), 0)
        del self.X_, self.y_idx_  # 長完就不需要了（森林有很多棵，省記憶體）
        return self

    def _grow(self, idx: np.ndarray, depth: int) -> Node:
        counts = np.bincount(self.y_idx_[idx], minlength=self.n_classes_).astype(float)
        node = Node(len(idx), counts, float(gini(counts)), depth)
        if node.impurity <= 0 or len(idx) < 2 * self.min_samples_leaf or (self.max_depth is not None and depth >= self.max_depth):
            return node
        gain, j, s = self.best_split(self.X_[idx], self.y_idx_[idx])
        if j < 0:
            return node
        node.feature, node.threshold = j, s
        go_left = self.X_[idx, j] <= s
        node.left, node.right = self._grow(idx[go_left], depth + 1), self._grow(idx[~go_left], depth + 1)
        return node

    def apply_one(self, x: np.ndarray) -> Node:
        node = self.root_
        while not node.is_leaf:
            node = node.left if x[node.feature] <= node.threshold else node.right
        return node

    def predict_proba(self, X: ArrayLike) -> np.ndarray:
        """(n, K)：葉子的類別比例。"""
        X = np.asarray(X, dtype=float)
        return np.array([(leaf := self.apply_one(x)).value / leaf.n_samples for x in X])

    def predict(self, X: ArrayLike) -> np.ndarray:
        return self.classes_[self.predict_proba(X).argmax(axis=1)]

    def nodes(self) -> list[Node]:
        out, stack = [], [self.root_]
        while stack:
            node = stack.pop()
            out.append(node)
            if not node.is_leaf:
                stack += [node.right, node.left]
        return out

    @property
    def feature_importances_(self) -> np.ndarray:
        imp = np.zeros(self.n_features_in_)
        for node in self.nodes():
            if not node.is_leaf:
                imp[node.feature] += node.n_samples * node.impurity - node.left.n_samples * node.left.impurity - node.right.n_samples * node.right.impurity
        return imp / imp.sum() if imp.sum() > 0 else imp


# 手寫版用的子樣本
N_SUB = 5000
rng_sub = np.random.default_rng(0)
sub = rng_sub.choice(len(Xtr), N_SUB, replace=False)
Xs, ys = Xtr[sub], ytr[sub]

import time

for mf in [None, "sqrt"]:
    t0 = time.perf_counter()
    tree = DecisionTree(max_features=mf, random_state=0).fit(Xs, ys)
    print(f"max_features={mf!s:<5}：fit {time.perf_counter() - t0:.2f} s、葉子 {sum(n.is_leaf for n in tree.nodes())}、test acc {np.mean(tree.predict(Xte) == yte):.4f}")
sk_d4 = DecisionTreeClassifier(max_depth=4, random_state=0).fit(Xs, ys)
my_d4 = DecisionTree(max_depth=4).fit(Xs, ys)
print("max_features=None、深度 4：與 sklearn 的 predict_proba 最大差異", np.abs(my_d4.predict_proba(Xte) - sk_d4.predict_proba(Xte)).max(), "（通常 = 0；不是 0 時見 TODO 8 (1)(c)）")

### 3B. Bootstrap 與 OOB

每棵樹 $b$：

1. 抽 bootstrap 樣本：`idx_b = rng.integers(0, N, size=N)`（列號，可能重複）；
2. OOB 遮罩：`oob_b[i] = True` ⇔ 第 $i$ 筆**一次都沒被抽到**。

**OOB 分數**：對每一筆訓練資料 $i$，只用「沒看過它」的那些樹（約 $0.368 B$ 棵）平均出一個預測 —— 這是一個不花額外訓練成本的驗證分數，效果接近交叉驗證。

#### 符號表（3B）

| 符號 | 意義 | shape | 程式裡 |
| --- | --- | --- | --- |
| idx | bootstrap 抽到的列號（可重複） | $(N,)$ | `idx` |
| counts | 每一筆被抽到幾次 | $(N,)$ | `np.bincount(idx, minlength=N)` |
| oob | 每一筆是不是這棵樹的 OOB | $(N,)$ bool | `oob` |

#### 最小例子：TODO 3（bootstrap 與 OOB 遮罩）

$N = 6$，抽一次 bootstrap。

1. **正確答案**：用迴圈一個一個抽、再數哪幾筆沒被抽到。
2. **積木**：`rng.integers(0, N, size=N)`、`np.bincount(..., minlength=N)`；以及 `np.isin` 的另一種寫法。
3. **留給你**：組出 `(idx, oob)`。

In [ ]:
# 最小例子：TODO 3（bootstrap 與 OOB 遮罩）
N_mini = 6

# ---- 1. 正確答案：迴圈 ----
rng_a = np.random.default_rng(1)
idx_loop = [int(rng_a.integers(0, N_mini)) for _ in range(N_mini)]
oob_loop = [i not in idx_loop for i in range(N_mini)]
print("抽到的列號 =", idx_loop, "\nOOB =", oob_loop)

# ---- 2. 積木 ----
rng_b = np.random.default_rng(1)
idx_vec = rng_b.integers(0, N_mini, size=N_mini)
print("\nrng.integers(0, N, size=N) =", idx_vec, "← 同一個 seed、同樣的抽法 → 跟迴圈一樣")
print("np.bincount(idx, minlength=N) =", np.bincount(idx_vec, minlength=N_mini), "← 每一筆被抽到幾次")
print("錯誤示範：np.bincount([0, 0, 2]) =", np.bincount([0, 0, 2]), "← N = 4 時第 3 筆沒被抽到，長度只有 3；minlength=4 →", np.bincount([0, 0, 2], minlength=4))

# ---- 3. 留給你 ----
# 問題：oob = (哪個積木) == 0？
# 寫完 TODO、執行過下一格之後，取消下面的註解來對照：
# print(bootstrap_sample(np.random.default_rng(1), N_mini))

In [ ]:
def bootstrap_sample(rng: np.random.Generator, N: int) -> tuple[np.ndarray, np.ndarray]:
    """回傳 (idx (N,)：抽到的列號、可重複, oob (N,) bool：沒被抽到的)。"""
    # TODO 3（程式）：bootstrap 與 OOB
    #   - idx = rng.integers(0, N, size=N)
    #   - oob = 每一筆被抽到的次數 == 0；次數用 np.bincount(idx, minlength=N)（minlength 不能省）
    raise NotImplementedError("TODO 3：bootstrap")


rng_chk = np.random.default_rng(0)
fracs = [1 - bootstrap_sample(rng_chk, N_SUB)[1].mean() for _ in range(200)]
print(f"✅ bootstrap 樣本中「不同的資料」比例：平均 {np.mean(fracs):.4f}（應 ≈ 1 − e⁻¹ = {1 - np.exp(-1):.4f}）")
idx_c, oob_c = bootstrap_sample(np.random.default_rng(0), N_SUB)
print("✅ OOB 的列沒有出現在 idx 裡：", not np.isin(np.flatnonzero(oob_c), idx_c).any(), "；不是 OOB 的列都有出現：", np.isin(np.flatnonzero(~oob_c), idx_c).all())

### 3C. 森林

**訓練**：for $b = 1..B$：抽 bootstrap → 用那些資料長一棵 `DecisionTree(max_features=m, random_state=…)` → 存下樹與它的 OOB 遮罩（TODO 5）。每棵樹的 `random_state` 要**不一樣**（否則每棵樹挑特徵的順序都一樣）。

**預測**：$\hat p(x) = \frac{1}{B}\sum_b \hat p_b(x)$ —— 平均**機率**（soft voting，sklearn 也是），再取 argmax（TODO 6）。

**OOB 分數**（TODO 7）：

$$\hat p^\text{OOB}(x_i) = \frac{\sum_b [i \in \text{OOB}_b]\ \hat p_b(x_i)}{\sum_b [i \in \text{OOB}_b]}$$

分子、分母各用一個累加器（前置複習 10）。$B$ 很小時，可能有少數資料「每一棵樹都抽到它」→ 分母 0 → 沒有 OOB 預測（設成 `nan`，算分數時排除）。

#### 符號表（3C）

| 符號 | 意義 | shape | 程式裡 |
| --- | --- | --- | --- |
| `trees_` | $B$ 棵樹（物件的串列） | 長度 $B$ | `self.trees_` |
| `oob_masks_` | 每棵樹的 OOB 遮罩 | $B$ 個 $(N,)$ | `self.oob_masks_` |
| $\hat p_b(X)$ | 第 $b$ 棵樹的機率 | $(n, K)$ | `tree.predict_proba(X)` |
| total | OOB 機率的累加 | $(N, K)$ | `total` |
| count | 每一筆當過幾次 OOB | $(N,)$ | `count` |
| `oob_proba_` | OOB 機率；沒有 OOB 樹的列是 `nan` | $(N, K)$ | `self.oob_proba_` |

#### 程式結構圖

```
forest = RandomForest(n_estimators=50, max_features="sqrt", random_state=0).fit(Xs, ys)
├── 超參數：n_estimators、max_features、min_samples_leaf、max_depth、bootstrap、random_state
├── classes_ (K,)
├── trees_      = [DecisionTree, DecisionTree, ...]   ← 每棵都有自己的 root_、_rng
├── oob_masks_  = [(N,) bool, (N,) bool, ...]         ← 跟 trees_ 一一對應
└── oob_proba_  (N, K)、oob_score_

fit：for b in range(B) → bootstrap_sample → DecisionTree(...).fit(X[idx], y[idx]) → append    ← TODO 5
predict_proba(X)：每棵樹 predict_proba → 疊起來平均                                           ← TODO 6
_compute_oob(X)：for tree, oob in zip(trees_, oob_masks_) → 只在 oob 的列上累加                ← TODO 7
```

`for tree in self.trees_:` 拿到的是樹物件本身；`tree.predict_proba` 呼叫的是那一棵樹的方法（`self` 在那裡指的是那棵樹，不是森林）。

#### 最小例子：TODO 6、7（平均與 OOB 累加）

3 棵樹、4 筆訓練資料、$K = 2$。每棵樹對 4 筆的機率（第 1 類）與 OOB 遮罩：

| 樹 | $\hat p_b$（第 1 類） | OOB 遮罩 |
| --- | --- | --- |
| 0 | (0.9, 0.2, 0.6, 0.1) | (T, F, F, F) |
| 1 | (0.7, 0.4, 0.8, 0.3) | (T, T, F, F) |
| 2 | (0.5, 0.0, 1.0, 0.5) | (F, T, F, F) |

1. **正確答案**：森林機率 = 三棵平均 → (0.7, 0.2, 0.8, 0.3)；OOB：第 0 筆用樹 0、1 → 0.8；第 1 筆用樹 1、2 → 0.2；第 2、3 筆沒有 OOB 樹 → nan。
2. **積木**：`np.stack(...).mean(axis=0)`、`total += p * oob[:, None]`、`count += oob`。
3. **留給你**：寫進 `predict_proba` 與 `_compute_oob`。

In [ ]:
# 最小例子：TODO 6、7（平均與 OOB 累加）
p1_mini = np.array([[0.9, 0.2, 0.6, 0.1], [0.7, 0.4, 0.8, 0.3], [0.5, 0.0, 1.0, 0.5]])  # (B=3, N=4) 第 1 類的機率
P_mini = np.stack([1 - p1_mini, p1_mini], axis=-1)  # (B, N, K=2)
oob_mini = np.array([[1, 0, 0, 0], [1, 1, 0, 0], [0, 1, 0, 0]], dtype=bool)  # (B, N)

# ---- 1. 正確答案：迴圈 ----
print("森林機率（第 1 類）：", [round(float(np.mean(p1_mini[:, i])), 4) for i in range(4)])
oob_ans = []
for i in range(4):
    used = [p1_mini[b, i] for b in range(3) if oob_mini[b, i]]
    oob_ans.append(round(float(np.mean(used)), 4) if used else float("nan"))
print("OOB 機率（第 1 類）：", oob_ans)

# ---- 2. 積木 ----
print("\nnp.stack([3 個 (4, 2)]).shape =", np.stack(list(P_mini)).shape, "→ .mean(axis=0) =", np.stack(list(P_mini)).mean(axis=0).shape)
total_mini, count_mini = np.zeros((4, 2)), np.zeros(4)
total_mini += P_mini[0] * oob_mini[0][:, None]
count_mini += oob_mini[0]
print("加完第 0 棵樹：total[:, 1] =", total_mini[:, 1], " count =", count_mini, "← oob[:, None] 是 (N, 1)，跟 (N, K) 廣播")
with np.errstate(invalid="ignore"):
    print("total / count[:, None]：0/0 → nan", (np.zeros((1, 2)) / np.zeros(1)[:, None]))

# ---- 3. 留給你 ----
# 問題：predict_proba 要對哪一軸平均？_compute_oob 的迴圈走過哪兩個串列（zip）？
# 寫完 TODO 5–7、執行過下一格之後，取消下面的註解來對照（用假樹 FixedTree 代替真的樹）：
# f_mini = RandomForest(); f_mini.classes_ = np.array([0, 1]); f_mini.trees_ = [FixedTree(P_mini[b]) for b in range(3)]; f_mini.oob_masks_ = list(oob_mini)
# X_rows = np.arange(4)[:, None]; print(f_mini.predict_proba(X_rows)[:, 1].round(4), f_mini._compute_oob(X_rows)[:, 1].round(4))


class FixedTree:
    """測試用的假樹：輸入 X 的第一欄當作「列號」，回傳固定機率表 (N, K) 的那幾列。"""

    def __init__(self, proba: np.ndarray) -> None:
        self.proba = proba

    def predict_proba(self, X) -> np.ndarray:
        return self.proba[np.asarray(X)[:, 0].astype(int)]

In [ ]:
from sklearn.metrics import accuracy_score, roc_auc_score


class RandomForest:
    """Random Forest 分類器：bootstrap + 每個節點隨機挑特徵 + 平均機率。"""

    def __init__(self, n_estimators: int = 50, max_features: int | Literal["sqrt"] | None = "sqrt", min_samples_leaf: int = 1, max_depth: int | None = None, bootstrap: bool = True, random_state: int = 0) -> None:
        self.n_estimators = n_estimators
        self.max_features = max_features
        self.min_samples_leaf = min_samples_leaf
        self.max_depth = max_depth
        self.bootstrap = bootstrap
        self.random_state = random_state

    def fit(self, X: ArrayLike, y: ArrayLike) -> Self:
        X, y = np.asarray(X, dtype=float), np.asarray(y)
        N = len(y)
        self.classes_ = np.unique(y)
        rng = np.random.default_rng(self.random_state)
        self.trees_, self.oob_masks_ = [], []
        for b in range(self.n_estimators):
            if self.bootstrap:
                idx, oob = bootstrap_sample(rng, N)
            else:  # 不抽樣：每棵樹都用全部資料，沒有 OOB
                idx, oob = np.arange(N), np.zeros(N, dtype=bool)
            # TODO 5（程式）：長第 b 棵樹並存起來
            #   - DecisionTree(max_depth=..., min_samples_leaf=..., max_features=..., random_state=...) 用森林的超參數
            #   - random_state 每棵樹要不一樣：例如 self.random_state * 10_000 + b
            #   - 用 X[idx]、y[idx] fit；把樹 append 到 self.trees_、oob append 到 self.oob_masks_
            raise NotImplementedError("TODO 5：長一棵樹")
        if self.bootstrap:
            self.oob_proba_ = self._compute_oob(X)
            has = ~np.isnan(self.oob_proba_[:, 0])
            self.oob_score_ = accuracy_score(y[has], self.classes_[self.oob_proba_[has].argmax(axis=1)])
        return self

    def predict_proba(self, X: ArrayLike) -> np.ndarray:
        """(n, K)：所有樹的機率平均。（假設每棵樹都看過全部 K 類；本章的資料抽 bootstrap 一定兩類都有）"""
        # TODO 6（程式）：平均所有樹的 predict_proba
        #   - [tree.predict_proba(X) for tree in self.trees_] → 疊起來 (B, n, K) → 對 B 那一軸平均
        raise NotImplementedError("TODO 6：平均")

    def predict(self, X: ArrayLike) -> np.ndarray:
        return self.classes_[self.predict_proba(X).argmax(axis=1)]

    def _compute_oob(self, X: np.ndarray) -> np.ndarray:
        """(N, K)：每一筆訓練資料只用「沒看過它」的樹平均出來的機率；沒有這種樹的列是 nan。"""
        N, K = len(X), len(self.classes_)
        total, count = np.zeros((N, K)), np.zeros(N)
        # TODO 7（程式）：OOB 累加
        #   - for tree, oob in zip(self.trees_, self.oob_masks_):
        #   - 只需要預測 oob 的那些列（X[oob]），省時間：total[oob] += tree.predict_proba(X[oob])；count[oob] += 1
        #     （或預測全部再乘上 oob[:, None]，結果一樣但慢）
        raise NotImplementedError("TODO 7：OOB")
        with np.errstate(invalid="ignore", divide="ignore"):
            return total / count[:, None]

    @property
    def feature_importances_(self) -> np.ndarray:
        return np.mean([t.feature_importances_ for t in self.trees_], axis=0)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# 檢查 1：不抽樣、不挑特徵 → 每棵樹都一模一樣 → 森林 = 一棵樹
same_forest = RandomForest(n_estimators=3, max_features=None, bootstrap=False, max_depth=6).fit(Xs, ys)
one_tree = DecisionTree(max_depth=6).fit(Xs, ys)
print("✅ bootstrap=False、max_features=None：森林與單棵樹的 predict_proba 最大差異",
      np.abs(same_forest.predict_proba(Xte) - one_tree.predict_proba(Xte)).max(), "（應 < 1e-12）")

# 檢查 2：手寫森林 vs sklearn（同樣的子樣本、同樣的超參數）—— 隨機性不同，只能比「統計上」接近
t0 = time.perf_counter()
forest = RandomForest(n_estimators=50, max_features="sqrt", random_state=0).fit(Xs, ys)
t_fit = time.perf_counter() - t0
p_mine = forest.predict_proba(Xte)[:, 1]
sk_rf = RandomForestClassifier(n_estimators=50, max_features="sqrt", random_state=0, oob_score=True, n_jobs=-1).fit(Xs, ys)
p_sk = sk_rf.predict_proba(Xte)[:, 1]
cmp = pd.DataFrame(
    {
        "scratch (50 trees)": [accuracy_score(yte, p_mine >= 0.5), roc_auc_score(yte, p_mine), forest.oob_score_],
        "sklearn (50 trees)": [accuracy_score(yte, p_sk >= 0.5), roc_auc_score(yte, p_sk), sk_rf.oob_score_],
    },
    index=["test accuracy", "test ROC AUC", "OOB accuracy"],
)
display(cmp.round(4))
print(f"手寫森林 fit {t_fit:.1f} s；兩者 test 機率的相關係數 {np.corrcoef(p_mine, p_sk)[0, 1]:.3f}")
print("✅ test accuracy 差距應 < 0.01：", abs(cmp.iloc[0, 0] - cmp.iloc[0, 1]) < 0.01)

# 檢查 3：OOB 分數 vs 5-fold CV（同一份子樣本）
cv_acc = cross_val_score(RandomForestClassifier(n_estimators=50, random_state=0, n_jobs=-1), Xs, ys, cv=cv).mean()
print(f"OOB accuracy（手寫）{forest.oob_score_:.4f} vs 5-fold CV（sklearn）{cv_acc:.4f}；沒有 OOB 預測的列：{int(np.isnan(forest.oob_proba_[:, 0]).sum())}")

In [ ]:
# TODO 8（問答）
#
# (1) 檢查 1：bootstrap=False、max_features=None 時，森林為什麼「應該」等於一棵樹？
#     _candidate_features 在 m = d 時直接回傳 np.arange(d)，沒有用亂數。
#     (a) 如果改成 m = d 時也呼叫 rng.choice(d, d, replace=False)（只是打亂順序），檢查 1 還會是 0 嗎？
#         （提示：三棵樹的 random_state 不同；第五章 TODO 11 (1)：平手時保留「先看到的」特徵 —— Adult 的特徵大多是整數或類別編碼，常常平手）
#     (b) sklearn 的 RandomForestClassifier(max_features=None, bootstrap=False) 每棵樹也一樣嗎？（sklearn 一律打亂特徵順序）
#     (c) 3A 最後跟 sklearn 比深度 4 的樹，如果差異不是 0，最可能的原因是什麼？
# 答：
#
# (2) 手寫 vs sklearn：
#     (a) 兩者的 test accuracy、AUC 差多少？為什麼不能要求「完全一樣」（第五章、第八章的檢查都可以）？
#     (b) 兩者的機率相關係數多高？這說明了什麼？
# 答：
#
# (3) OOB：
#     (a) OOB accuracy 和 5-fold CV 差多少？
#     (b) OOB 用了幾棵樹來預測每一筆？（大約 0.368 × B）。用比較少的樹平均，OOB 會比「真的」森林偏好還是偏差？
#     (c) 為什麼 OOB 對 Random Forest 是「免費」的，對第三章的邏輯回歸卻不是？（對照第四章 TODO 10 (4) 的 LOO 討論）
# 答：

### 3D. 實驗：樹的棵數、隨機特徵與相關、變異、Extra Trees

這一節都用 sklearn 在**完整的訓練集**上做（比手寫版快很多）。

#### 本節名詞

| 名詞 | 定義 |
| --- | --- |
| `warm_start=True` | 再呼叫一次 `fit` 時保留已經長好的樹，只長新增的那幾棵 → 一次畫出「棵數 vs 分數」 |
| strength（單棵樹的強度） | 個別樹的平均 test accuracy |
| 樹之間的相關 $\bar\rho$ | 隨機挑兩棵樹，它們對 test 的預測機率的相關係數，取平均 |
| `estimators_` | sklearn 森林裡每一棵樹（`DecisionTreeClassifier`）的串列 |
| Extra Trees | 每個節點、每個候選特徵只**隨機挑一個切點**（不找最好的），再從這些隨機切點中選最好的；不做 bootstrap（預設） |

In [ ]:
# 實驗 1：樹的棵數。OOB error 隨 B 的變化（三種 max_features）
import warnings

from sklearn.ensemble import ExtraTreesClassifier

Bs = [5, 10, 20, 40, 80, 150, 300]
curves = {}
for mf, name in [(1, "max_features=1"), ("sqrt", "max_features=sqrt (4)"), (None, "max_features=None (bagging)")]:
    m = RandomForestClassifier(max_features=mf, warm_start=True, oob_score=True, random_state=0, n_jobs=-1)
    errs = []
    for B in Bs:
        m.set_params(n_estimators=B)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")  # B 很小時有些樣本沒有 OOB 預測
            m.fit(Xtr, ytr)
        errs.append(1 - m.oob_score_)
    curves[name] = errs
curve_tbl = pd.DataFrame(curves, index=Bs).rename_axis("n_estimators")
fig, ax = plt.subplots(figsize=(7, 4))
for name in curve_tbl:
    ax.plot(Bs, curve_tbl[name], "o-", label=name)
ax.set_xscale("log")
ax.set_xlabel("number of trees (log)")
ax.set_ylabel("OOB error")
ax.set_title("OOB error vs number of trees")
ax.legend()
plt.show()
display(curve_tbl.round(4))

In [ ]:
# TODO 9（問答）樹的棵數
#
# (1) (a) OOB error 隨 B 增加怎麼變？B 從 150 到 300 還有進步嗎？
#     (b) 有沒有看到「樹太多 → 過擬合」？用前置複習 2 解釋：B 增加只會讓哪一項變小？
#     (c) 所以 n_estimators 要「調」嗎？它應該當作「越大越好、直到不划算」的參數，還是像 max_depth 那樣有最佳值？
#         它的代價是什麼？
# 答：
#
# (2) 三條線的終點：max_features=1、sqrt、None（bagging）誰最好？差多少？（實驗 2 會解釋）
# 答：

In [ ]:
# 實驗 2：max_features → 單棵樹的強度、樹之間的相關、森林的表現
rng_e = np.random.default_rng(0)
rows = []
for mf in [1, 2, 3, 4, 6, 9, 14]:
    m = RandomForestClassifier(n_estimators=100, max_features=mf, random_state=0, n_jobs=-1, oob_score=True).fit(Xtr, ytr)
    P = np.array([t.predict_proba(Xte)[:, 1] for t in m.estimators_[:40]])  # 40 棵樹 × n_test
    C = np.corrcoef(P)
    rho = C[np.triu_indices_from(C, k=1)].mean()
    rows.append(
        {
            "max_features m": mf,
            "single-tree acc (strength)": np.mean([accuracy_score(yte, p >= 0.5) for p in P]),
            "mean pairwise ρ": rho,
            "forest OOB acc": m.oob_score_,
            "forest test AUC": roc_auc_score(yte, m.predict_proba(Xte)[:, 1]),
        }
    )
mf_tbl = pd.DataFrame(rows).set_index("max_features m")
display(mf_tbl.round(4))
fig, ax1 = plt.subplots(figsize=(7, 4))
ax1.plot(mf_tbl.index, mf_tbl["single-tree acc (strength)"], "o-", label="single-tree accuracy")
ax1.plot(mf_tbl.index, mf_tbl["forest OOB acc"], "s-", label="forest OOB accuracy")
ax1.set_xlabel("max_features m")
ax1.set_ylabel("accuracy")
ax2 = ax1.twinx()
ax2.plot(mf_tbl.index, mf_tbl["mean pairwise ρ"], "^--", color="C2", label="mean ρ between trees")
ax2.set_ylabel("ρ", color="C2")
fig.legend(loc="lower right", bbox_to_anchor=(0.88, 0.15))
ax1.set_title("Strength vs correlation")
plt.show()

In [ ]:
# TODO 10（問答）隨機特徵：強度 vs 相關
#
# (1) 讀表：m 從 1 增加到 14（= bagging），
#     (a) 單棵樹的 accuracy（強度）怎麼變？為什麼？（每個節點能挑的特徵越多，那一刀越好）
#     (b) 樹之間的相關 ρ 怎麼變？為什麼？（提示：EDA 看過 relationship／marital-status、capital-gain 很強；
#         m = 14 時每棵樹的根節點都會選誰？m = 1 呢？）
#     (c) 森林的表現在哪個 m 附近最好？
# 答：
#
# (2) 用前置複習 2 的公式 Var = ρσ² + (1−ρ)σ²/B 解釋 (1)：
#     森林的錯誤來自兩件事 —— 每棵樹的偏差（強度不夠）、平均之後剩下的變異（≈ ρσ²）。
#     m 小：ρ 小（好）但每棵樹弱（壞）；m 大：反過來。所以 m 是在平衡什麼？
#     （Breiman 2001 的結論：森林的泛化誤差上界 ∝ ρ̄ / s²，s 是強度）
# 答：
#
# (3) 第五章實驗 4 的 20 棵 bootstrap 樹「根節點都一樣」。那個實驗其實是 m 等於多少的森林？
#     這解釋了為什麼那裡的平均只把 accuracy 從 0.746 拉到 0.790、而不是更多嗎？
# 答：

In [ ]:
# 實驗 3：變異。在 8 份「互不重疊」的訓練子集（各 4,000 筆）上分別訓練，看 test 預測有多不穩定
from sklearn.ensemble import BaggingClassifier

perm = np.random.default_rng(1).permutation(len(Xtr))
chunks = np.array_split(perm[: 8 * 4000], 8)
models = {
    "single tree (fully grown)": lambda: DecisionTreeClassifier(random_state=0),
    "single tree (max_depth=8)": lambda: DecisionTreeClassifier(max_depth=8, random_state=0),
    "bagging (100 trees)": lambda: BaggingClassifier(DecisionTreeClassifier(), n_estimators=100, random_state=0, n_jobs=-1),
    "random forest (100 trees)": lambda: RandomForestClassifier(n_estimators=100, random_state=0, n_jobs=-1),
}
rows = []
for name, make in models.items():
    P = np.array([make().fit(Xtr[c], ytr[c]).predict_proba(Xte)[:, 1] for c in chunks])  # (8, n_test)
    H = P >= 0.5
    rows.append(
        {
            "model": name,
            "mean test acc": np.mean([accuracy_score(yte, h) for h in H]),
            "mean test AUC": np.mean([roc_auc_score(yte, p) for p in P]),
            "prediction disagreement (pairs)": np.mean([np.mean(H[a] != H[b]) for a in range(8) for b in range(a + 1, 8)]),
            "variance of P(>50K) across subsets": P.var(axis=0).mean(),
        }
    )
display(pd.DataFrame(rows).set_index("model").round(4))

# 實驗 4：RF vs Extra Trees vs Bagging（5-fold CV，完整訓練集）
for name, m in {
    "Bagging (100 trees)": BaggingClassifier(DecisionTreeClassifier(), n_estimators=100, random_state=0, n_jobs=-1),
    "Random Forest (100 trees)": RandomForestClassifier(n_estimators=100, random_state=0, n_jobs=-1),
    "Extra Trees (100 trees)": ExtraTreesClassifier(n_estimators=100, random_state=0, n_jobs=-1),
}.items():
    baseline.loc[name] = cv_summary(Pipeline([("prep", make_prep()), ("m", m)]))
display(baseline)

In [ ]:
# TODO 11（問答）變異與 Extra Trees
#
# (1) 實驗 3：
#     (a) 長滿的單棵樹：在不同的訓練子集上，兩兩預測不一致的比例是多少？森林呢？「P(>50K) 的變異數」差幾倍？
#     (b) 深度 8 的單棵樹也比長滿的穩定很多，但它的 AUC 比森林低。用偏差—變異說明：
#         「剪枝」和「平均」各是怎麼降低變異的？哪一個會付出偏差的代價？（前置複習 6）
# 答：
#
# (2) 實驗 4（看 baseline 表的最後三列）：
#     (a) Bagging、Random Forest、Extra Trees 的 AUC 各多少？fit 時間呢？
#     (b) Extra Trees 每個節點不找最好的切點、而是隨機切。這會讓單棵樹更強還是更弱？樹之間更像還是更不像？
#         為什麼它還能跟 RF 差不多好，而且比較快？
# 答：

---
## 4. 套件實作：sklearn `RandomForestClassifier`

#### 本節名詞

| 名詞 | 意思 |
| --- | --- |
| `n_jobs=-1` | 用所有 CPU 核心平行長樹（樹與樹互相獨立，平行化非常容易 —— 第 12 章的 boosting 就不行） |
| `oob_score=True` | 訓練完附帶算 OOB accuracy（`oob_score_`）與 OOB 機率（`oob_decision_function_`） |
| `feature_importances_` | 每棵樹的 impurity importance（第五章 TODO 10）的平均 |
| `permutation_importance` | 在**測試集**上把一欄打亂，看 AUC 掉多少（第五章 TODO 20） |
| `class_weight="balanced"` | 少數類的樣本權重放大（本章不調，見待補強） |

#### 程式結構圖

```
rf_pipe = Pipeline([("prep", make_prep()), ("rf", RandomForestClassifier(n_estimators=300, oob_score=True, n_jobs=-1))])
├── rf_pipe["prep"]：ColumnTransformer（缺值 → "Missing"、OrdinalEncoder）
└── rf_pipe["rf"]  ：estimators_（300 棵 DecisionTreeClassifier）、oob_score_、oob_decision_function_、feature_importances_
參數名：rf__max_features、rf__min_samples_leaf、rf__max_depth、rf__n_estimators
```

In [ ]:
from sklearn.inspection import permutation_importance

rf_pipe = Pipeline([("prep", make_prep()), ("rf", RandomForestClassifier(n_estimators=300, oob_score=True, random_state=0, n_jobs=-1))])
t0 = time.perf_counter()
rf_pipe.fit(X_train, y_train)
print(f"fit {time.perf_counter() - t0:.1f} s；OOB accuracy {rf_pipe['rf'].oob_score_:.4f}；test accuracy {rf_pipe.score(X_test, y_test):.4f}")
print("每棵樹的葉子數（中位數）：", int(np.median([t.get_n_leaves() for t in rf_pipe["rf"].estimators_])), "；深度（中位數）：", int(np.median([t.get_depth() for t in rf_pipe["rf"].estimators_])))

perm = permutation_importance(rf_pipe, X_test, y_test, scoring="roc_auc", n_repeats=5, random_state=0)  # 模型本身已經用 n_jobs=-1
imp = pd.DataFrame(
    {
        "impurity (train)": pd.Series(rf_pipe["rf"].feature_importances_, index=FEATURES),
        "permutation AUC drop (test)": pd.Series(perm.importances_mean, index=X_test.columns).reindex(FEATURES),
        "scratch forest impurity (5k subsample)": pd.Series(forest.feature_importances_, index=FEATURES),
        "unique values": Z_train.nunique(),
    }
).sort_values("impurity (train)", ascending=False)
display(imp.round(4))
fig, ax = plt.subplots(figsize=(8, 5))
imp[["impurity (train)", "permutation AUC drop (test)"]].iloc[::-1].plot.barh(ax=ax)
ax.set_title("Impurity vs permutation importance")
plt.show()

In [ ]:
# TODO 12（問答）特徵重要度
#
# (1) (a) impurity importance 的前三名是哪些？permutation importance 的前三名呢？
#     (b) fnlwgt 在兩種重要度中各排第幾？跟 TODO 1 (2)(b) 的預測一樣嗎？用「不同值的個數」解釋。
#     (c) 森林已經平均了很多棵樹，為什麼 impurity importance 還是會被 fnlwgt 騙？（提示：每棵樹都是長滿的；平均不會修正「每棵都偏」的偏差）
# 答：
#
# (2) relationship 和 marital-status 高度相關（TODO 1 (2)(a)）。
#     (a) 兩者的 permutation importance 各是多少？如果把其中一欄打亂，模型還能從另一欄拿到差不多的資訊嗎？
#     (b) 所以「permutation importance 低」一定代表「這個特徵沒用」嗎？
# 答：
#
# (3) 每棵樹的中位數葉子數有多少？跟第五章 Titanic 的樹比呢？這樣一片森林，還能像第五章那樣「把規則印出來給人看」嗎？
#     森林用什麼代價換到了比較好的分數？
# 答：

---
## 5. 調參

全程只用 `X_train`。README：Random Forest 以後改用 **random search**（參數多、每次 fit 不便宜）。另外，RF 有一個便宜的替代驗證：**OOB**。

| 類別 | 超參數 | 說明 |
| --- | --- | --- |
| 隨機性 | `max_features` | 每個節點挑幾個特徵（實驗 2）；最重要的參數 |
| 容量 | `min_samples_leaf` | 葉子最少幾筆；RF 的樹通常長滿，但葉子大一點可以讓機率比較平滑 |
| 容量 | `max_depth` | 通常不限 |
| 不用調 | `n_estimators` | 越多越好、只是越慢（TODO 9）；調參時固定 200，最後再用多一點 |

**選模型的指標用 ROC AUC**（不平衡資料，而且之後可以再選門檻）。

#### 本節名詞

- **`RandomizedSearchCV(n_iter=…)`**：從參數分布中隨機抽 `n_iter` 組來試，不是全部試完。參數多時，比 grid 有效率（Bergstra & Bengio 2012：真正重要的參數通常只有一兩個，random search 在那幾個方向上試到的值比較多）。
- **OOB 調參**：每組參數只 fit 一次（不用 5 次），用 `oob_decision_function_` 算 AUC。

In [ ]:
from scipy.stats import randint
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    "rf__max_features": [1, 2, 3, 4, 5, 6, 8, 10, 14],
    "rf__min_samples_leaf": randint(1, 40),
    "rf__max_depth": [None, 10, 15, 20, 30],
}
rs = RandomizedSearchCV(
    Pipeline([("prep", make_prep()), ("rf", RandomForestClassifier(n_estimators=200, random_state=0, n_jobs=-1))]),
    param_dist, n_iter=20, cv=cv, scoring="roc_auc", random_state=0, n_jobs=1,
)
t0 = time.perf_counter()
rs.fit(X_train, y_train)
print(f"random search：{time.perf_counter() - t0:.0f} s；best_params_：{rs.best_params_}；CV AUC {rs.best_score_:.4f}")
rs_res = pd.DataFrame(rs.cv_results_)[["param_rf__max_features", "param_rf__min_samples_leaf", "param_rf__max_depth", "mean_test_score", "std_test_score", "mean_fit_time"]]
rs_res.columns = ["max_features", "min_samples_leaf", "max_depth", "mean AUC", "std", "fit s"]
display(rs_res.sort_values("mean AUC", ascending=False).round(4).head(10))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, col in zip(axes, ["max_features", "min_samples_leaf"]):
    ax.scatter(rs_res[col].astype(float), rs_res["mean AUC"], c=rs_res["max_depth"].fillna(50).astype(float), cmap="viridis")
    ax.set_xlabel(col)
    ax.set_ylabel("CV AUC")
axes[0].set_title("Random search results (color = max_depth, None shown as 50)")
plt.show()

#### 最小例子：TODO 13（用 OOB 機率算分數）

4 筆訓練資料、OOB 機率（第 1 類）`[0.8, nan, 0.3, 0.6]`、真實標籤 `[1, 0, 0, 1]`。

1. **正確答案**：排除 nan 那一筆，再算 AUC。目標：剩 3 筆 `(0.8, 1)、(0.3, 0)、(0.6, 1)` → 正例都高於負例 → AUC = 1.0。
2. **積木**：`np.isnan`、布林遮罩、`roc_auc_score`。
3. **留給你**：在迴圈裡對每組參數算 OOB AUC。

In [ ]:
# 最小例子：TODO 13（用 OOB 機率算分數）
oob_p_mini = np.array([0.8, np.nan, 0.3, 0.6])
y_oob_mini = np.array([1, 0, 0, 1])

# ---- 1. 正確答案 ----
keep_loop = [i for i in range(4) if not np.isnan(oob_p_mini[i])]
print("保留的列：", keep_loop, " AUC =", roc_auc_score(y_oob_mini[keep_loop], oob_p_mini[keep_loop]))

# ---- 2. 積木 ----
print("\n~np.isnan(p) =", ~np.isnan(oob_p_mini))
try:
    roc_auc_score(y_oob_mini, oob_p_mini)
except ValueError as err:
    print("直接丟進 roc_auc_score：ValueError", str(err)[:60], "...")

# ---- 3. 留給你 ----
# 問題：sklearn 的 oob_decision_function_ 是 (N, K)；第 1 類那一欄是 [:, 1]。怎麼排除 nan 再算 AUC？

In [ ]:
# OOB 調參：max_features × min_samples_leaf，每組只 fit 一次
Z_full = make_prep().fit_transform(X_train)
oob_rows = []
t0 = time.perf_counter()
for mf in [2, 3, 4, 6, 8]:
    for msl in [1, 3, 5, 10, 20]:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            m = RandomForestClassifier(n_estimators=200, max_features=mf, min_samples_leaf=msl, oob_score=True, random_state=0, n_jobs=-1).fit(Z_full, y_train)
        p_oob = m.oob_decision_function_[:, 1]
        # TODO 13（程式）：這組參數的 OOB AUC
        #   - keep = ~np.isnan(p_oob)；auc = roc_auc_score(y_train[keep], p_oob[keep])
        raise NotImplementedError("TODO 13：OOB AUC")
        oob_rows.append({"max_features": mf, "min_samples_leaf": msl, "OOB AUC": auc})
print(f"OOB 調參：{len(oob_rows)} 組，{time.perf_counter() - t0:.0f} s（每組 fit 1 次；5-fold CV 要 5 次）")
oob_res = pd.DataFrame(oob_rows)
heat = oob_res.pivot_table(index="max_features", columns="min_samples_leaf", values="OOB AUC")
fig, ax = plt.subplots(figsize=(7, 4))
sns.heatmap(heat, annot=True, fmt=".4f", cmap="viridis", ax=ax)
ax.set_title("OOB ROC AUC")
plt.show()
best_oob = oob_res.loc[oob_res["OOB AUC"].idxmax()]
print("OOB 最佳：", best_oob.to_dict())

In [ ]:
# TODO 14（問答）調參
#
# (1) random search：
#     (a) 前 10 名的 max_features、min_samples_leaf 有什麼共同點？max_depth 重要嗎？
#     (b) 前 10 名之間的 AUC 差多少？跟 std 比呢？
# 答：
#
# (2) OOB 調參：
#     (a) OOB 最佳的組合跟 random search 的最佳一致嗎？
#     (b) min_samples_leaf 從 1 變大，AUC 怎麼變？為什麼「葉子大一點」對 AUC 有幫助，但對單棵樹（第五章）的意義不一樣？
#         （提示：森林的機率 = 很多葉子比例的平均；每個葉子只有 1 筆時，每棵樹的機率只有 0 或 1）
#     (c) 用 OOB 而不是 CV 調參，省了多少時間？代價是什麼？
# 答：

---
## 6. 測試集評估

測試集只評估一次。對照組：**邏輯回歸**（第三章；數值欄標準化、類別欄 one-hot，$C = 1$）。

#### 指標

- **accuracy、precision／recall／F1**（以 >50K = 1 為正類）、**ROC AUC**。
- `log_result`／`compare`：成績寫進 `results/adult.csv`，依 ROC AUC 排序。

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from ml_dl_llm.metrics import classification_metrics
from ml_dl_llm.results import compare, log_result

BENCH = "adult"
logreg = Pipeline(
    [
        ("prep", ColumnTransformer([
            ("num", StandardScaler(), NUM),
            ("cat", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="Missing")), ("oh", OneHotEncoder(handle_unknown="ignore"))]), CAT),
        ])),
        ("lr", LogisticRegression(max_iter=3000)),
    ]
).fit(X_train, y_train)
tree_full = Pipeline([("prep", make_prep()), ("tree", DecisionTreeClassifier(random_state=0))]).fit(X_train, y_train)
tree_d8 = Pipeline([("prep", make_prep()), ("tree", DecisionTreeClassifier(max_depth=8, random_state=0))]).fit(X_train, y_train)
best_rf = rs.best_estimator_.set_params(rf__n_estimators=500).fit(X_train, y_train)  # 最後用多一點樹
oob_rf = Pipeline([("prep", make_prep()), ("rf", RandomForestClassifier(n_estimators=500, max_features=int(best_oob["max_features"]), min_samples_leaf=int(best_oob["min_samples_leaf"]), random_state=0, n_jobs=-1))]).fit(X_train, y_train)
et = Pipeline([("prep", make_prep()), ("et", ExtraTreesClassifier(n_estimators=300, random_state=0, n_jobs=-1))]).fit(X_train, y_train)
dummy = DummyClassifier(strategy="prior").fit(X_train, y_train)

# 名稱: (模型 or 機率, 參數)；手寫森林用 5k 子樣本
eval_models = {
    "Dummy prior": (dummy.predict_proba(X_test)[:, 1], {}),
    "Tree fully grown (sklearn)": (tree_full.predict_proba(X_test)[:, 1], {}),
    "Tree max_depth=8 (sklearn)": (tree_d8.predict_proba(X_test)[:, 1], {"max_depth": 8}),
    "RF 50 trees, 5k subsample (scratch)": (forest.predict_proba(Xte)[:, 1], {"n_estimators": 50, "n_train": N_SUB}),
    "RF 300 trees default (sklearn)": (rf_pipe.predict_proba(X_test)[:, 1], {"n_estimators": 300}),
    "RF random-search best (sklearn)": (best_rf.predict_proba(X_test)[:, 1], {k.split("__")[-1]: str(v) for k, v in rs.best_params_.items()}),
    "RF OOB-tuned (sklearn)": (oob_rf.predict_proba(X_test)[:, 1], {"max_features": int(best_oob["max_features"]), "min_samples_leaf": int(best_oob["min_samples_leaf"])}),
    "Extra Trees 300 (sklearn)": (et.predict_proba(X_test)[:, 1], {}),
    "LogReg C=1 (sklearn, ref)": (logreg.predict_proba(X_test)[:, 1], {"C": 1.0}),
}
test_scores = {}
for name, (prob, params) in eval_models.items():
    m = classification_metrics(y_test, (prob >= 0.5).astype(int), prob)
    test_scores[name] = m
    log_result(BENCH, name, m, **params)
display(pd.DataFrame(test_scores).T)
compare(BENCH, "roc_auc")

---
## 7. 診斷

#### 本節名詞

- **分組錯誤率**：依 `relationship`、`education-num` 分組，看哪一群錯最多。
- **信心分組**（第六章 TODO 12）：把預測依「模型說的機率」分組，比較平均機率與實際 >50K 的比例。
- **部分相依圖（partial dependence，PD）**：把所有人的某一個特徵都改成同一個值 $v$、其他特徵不變，平均模型的預測；$v$ 掃過一個範圍，畫成曲線。看的是「模型」學到的關係（不一定是因果）。

In [ ]:
from sklearn.inspection import PartialDependenceDisplay
from ml_dl_llm.metrics import plot_confusion_matrix

final = best_rf
p_te = final.predict_proba(X_test)[:, 1]
pred_te = (p_te >= 0.5).astype(int)
plot_confusion_matrix(y_test, pred_te, labels=["<=50K", ">50K"])
plt.title("Tuned RF on test (threshold 0.5)")
plt.show()

diag = X_test.assign(y=yte, p=p_te, correct=pred_te == yte)
display(diag.groupby("relationship", observed=True).agg(n=("y", "size"), true_rate=("y", "mean"), mean_p=("p", "mean"), accuracy=("correct", "mean")).round(3))

bins = pd.cut(p_te, [0, 0.1, 0.3, 0.5, 0.7, 0.9, 1.0], include_lowest=True)
display(pd.DataFrame({"p": p_te, "y": yte}).groupby(bins, observed=False).agg(n=("y", "size"), mean_p=("p", "mean"), actual_rate=("y", "mean")).round(3))

fig, ax = plt.subplots(figsize=(14, 3.5))
PartialDependenceDisplay.from_estimator(final, X_test.sample(2000, random_state=0), ["age", "education-num", "hours-per-week", "capital-gain"], ax=ax, n_cols=4, grid_resolution=30)
plt.show()

In [ ]:
# TODO 15（問答，本章總結）
#
# (1) 錯在哪裡：
#     (a) 混淆矩陣：錯誤主要是哪一種（把 >50K 猜成 <=50K，或反過來）？跟類別不平衡有什麼關係？要怎麼調整？（第六章的門檻）
#     (b) 依 relationship 分組：哪一組的 accuracy 最低？那一組的真實 >50K 比例是多少？為什麼「比例接近 0.5」的組最難？
#     (c) 信心分組表：模型說「0.9–1.0」的那一組，實際 >50K 的比例是多少？「0.0–0.1」呢？RF 的機率有沒有校準好？
# 答：
#
# (2) 部分相依圖：
#     (a) age 的曲線長什麼樣子？在哪個年紀最高？
#     (b) capital-gain 的曲線為什麼是「階梯」狀的？（提示：樹只能用「≤ 門檻」切）
#     (c) 這些曲線能說「多讀一年書會讓收入變高」嗎？
# 答：
#
# (3) 跟前幾章比（看第 6 節的表）：
#     (a) RF、單棵樹（兩種）、邏輯回歸的 test AUC 各多少？RF 比邏輯回歸好多少？
#     (b) 比較表（填空）：
#
#         |                        | Random Forest | 單棵決策樹 | 邏輯回歸 |
#         | 偏差／變異             |               |            |          |
#         | 需要調的參數           |               |            |          |
#         | 訓練能不能平行         |               |            |          |
#         | 可解釋性               |               |            |          |
#         | 免費的驗證             |               |            |          |
#
#     (c) RF 的「預設參數就很好」是它被廣泛使用的原因之一。從這一章的調參結果看，預設和調過的差多少？
# 答：

---
## 8. 本章小結

（完成所有 TODO、並經過回饋與修正之後，依 README「章末小結」填寫：問答註解轉成 markdown、結果表、核心觀念、待補強。）

### 方法與結果

Adult（train / test 依載入結果；CV = 5-fold 分層；主要指標 = ROC AUC）。

| 方法 | 做法 | 關鍵性質 | CV AUC | test AUC |
| --- | --- | --- | --- | --- |
| Dummy prior | | | | |
| 單棵樹（長滿／深度 8） | 第五章 | | | |
| RF（手寫，5k 子樣本） | bootstrap + 每節點隨機特徵 + 平均 | | — | |
| RF（sklearn 預設） | | | | |
| RF（random search／OOB 調參） | | | | |
| Extra Trees | | | | |
| 邏輯回歸（對照） | | | — | |

### 核心觀念

1.
2.
3.

### 待補強（之後回來可以學）

**理論**
- ☐ Breiman (2001) 的泛化誤差上界：強度 $s$ 與相關 $\bar\rho$
- ☐ 為什麼 RF「不會因為樹多而過擬合」，但仍可能因為樹太深而過擬合（min_samples_leaf 的角色）
- ☐ RF 與自適應最近鄰（adaptive nearest neighbors）的關係；proximity matrix

**特徵與解釋**
- ☐ 類別特徵的編碼：ordinal vs one-hot vs target encoding；sklearn 1.4 起 RF 原生支援類別特徵的條件
- ☐ 條件 permutation importance、drop-column importance、SHAP（TreeSHAP）
- ☐ ICE 曲線（individual conditional expectation）與 PD 的差別

**延伸**
- ☐ 不平衡資料：`class_weight="balanced_subsample"`、Balanced Random Forest
- ☐ 機率校準：`CalibratedClassifierCV`
- ☐ Quantile Regression Forest、Isolation Forest（第八章）
- ☐ 跟 boosting 的比較：bagging 降變異、boosting 降偏差（第 9、10 章）